# Rubrik adapter'ı — eğitim

Ürünün rubrik motorunu (`/analysis/run`) çalıştıran adapter. Tek adapter iki
rubriği birden öğreniyor: `startup-investability` (9 kriter) ve
`digital-marketing` (6 kriter). İkisi farklı kriterler soruyor ama **aynı
davranışı** istiyor — şemayı doldur, vakadan alıntıla, ve vaka bir kriterde
sessizse bunu söyle.

Temel modelin ölçülmüş hali:

| ölçüm | base | anlamı |
|---|---|---|
| `absent_rate` | **0** | kanıtı olmayan kritere yine de 3/5 veriyor |
| `schema_valid` | **0** | her cevabı ```json bloğuna sarıyor |
| `stddev_score` | 0 | tutarlılık zaten iyi — bozmamak lazım |

**Bu notebook ölçmüyor, eğitiyor.** Tam ölçüm `rubric-eval` notebook'unda ve
orada taban ile adapter *aynı oturumda* koşuyor — aynı kütüphane sürümleri,
kesin karşılaştırılabilir sayılar. Bir Kaggle oturumu 12 saatle sınırlı ve
ikisini bir arada koşmak sekiz saate dayanıyordu; ölçümün uzaması yüzünden beş
saatlik eğitimi kaybetmek istemiyoruz.

Aşağıdaki tek istisna **ucuz taban kapısı**: 20 satırlık hızlı bir kontrol.
Amacı sayı üretmek değil, eğitime hiç başlamamak gereken durumu yakalamak —
Flutter v8'de temel model işi zaten yapıyordu ve bu, eğitim bittikten sonra
anlaşıldı.

In [ ]:
import glob, json, os, shutil, sys
import torch

assert torch.cuda.is_available(), "GPU acik degil - Settings > Accelerator > GPU T4"
cap = torch.cuda.get_device_capability(0)
print("GPU:", torch.cuda.get_device_name(0), "sm_%d%d" % cap)
print("bellek: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

# Fail here, in five seconds, rather than after an 8 GB download. A P100 is
# sm_60: Kaggle's torch build does not support it at all, and bitsandbytes needs
# sm_75 for 4-bit NF4. The Flutter run landed on one because kernel-metadata
# omitted machine_shape, and the error arrived half an hour in wearing a
# different mask.
assert cap >= (7, 5), (
    f"sm_{cap[0]}{cap[1]} yetersiz - 4-bit NF4 icin T4 (sm_75) gerekiyor. "
    "Settings > Accelerator > GPU T4 x2")

In [ ]:
# Qwen3 icin transformers >= 4.51 gerekiyor; Kaggle imaji eskiyse sessizce
# 'unknown architecture' ile duser.
!pip -q install -U "transformers>=4.51" "peft>=0.11" "bitsandbytes>=0.43" "accelerate>=0.30" datasets 2>&1 | tail -2
import transformers, peft, bitsandbytes
print("transformers", transformers.__version__, "| peft", peft.__version__, "| bnb", bitsandbytes.__version__)

In [ ]:
def find_mount(slug, marker):
    """Locate one input mount by the dataset/kernel slug in its path.

    Not by filename. A kernel attached with kernel_sources contributes the whole
    of its /kaggle/working, which for the training run includes its own copies
    of the data files and the scripts — so searching for `rubric_eval.jsonl`
    finds two mounts and picks between them by luck. The slug is the only thing
    that distinguishes them, and it appears in the path.

    Recursive on top of that, because the mount depth is not a promise: the same
    dataset has appeared directly under /kaggle/input and, on the next run, one
    level deeper under /kaggle/input/datasets.
    """
    hits = [p for p in glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
            if slug.split("/")[-1] in p]
    assert hits, (f"'{slug}' bagli degil (aranan: {marker}). "
                  f"Kaggle > Notebook > Add Input, ve surumun islenmesi bitmis olmali.")
    return os.path.dirname(sorted(hits, key=len)[0])


for root, dirs, files in os.walk("/kaggle/input"):
    print(root, "->", sorted(files)[:4], "..." if len(files) > 4 else "")
    if root.count("/") > 6:
        dirs.clear()

In [ ]:
WORK = "/kaggle/working"
DATA = find_mount("emrahik/rubric-dataset", "rubric_train.jsonl")
print("veri seti:", DATA)

os.makedirs(f"{WORK}/data", exist_ok=True)
for f in os.listdir(DATA):
    dst = f"{WORK}/data/{f}" if f.endswith(".jsonl") else f"{WORK}/{f}"
    shutil.copy(f"{DATA}/{f}", dst)
os.chdir(WORK)
print(sorted(os.listdir(WORK)))
print(sorted(os.listdir(f"{WORK}/data")))

## 1. Taban kapısı — ucuz kontrol

20 satır, contrast yok. `absent_rate` ve `schema_valid` burada 0'a yakın
çıkmalı. Çıkmıyorsa **dur**: temel model işi zaten yapıyorsa adapter'ın
ölçülecek bir işi yoktur ve beş saat harcanmasın.

In [ ]:
!python rubric_eval.py --base-only \
    --data data/rubric_eval.jsonl \
    --base-model Qwen/Qwen3-4B-Instruct-2507 \
    --limit 20 \
    --out out/base_gate.json

## 2. Eğitim

1600 satır, 3 epoch, effective batch 16 → ~300 optimizer adımı.

`max-seq-len 2560` ölçülerek seçildi (`measure_tokens.py`): karışımın en uzun
satırı 2477 token, p95 2308. 2048 satırların %14'ünü kırpar ve kırpma soldan
olduğu için o satırlar cevabını korur, vakasının başını kaybeder — yani modele
hiç görmediği kanıta atıf yapmayı öğretir, gayet normal görünen bir loss'la.

In [ ]:
!python train_qlora_qwen.py \
    --train data/rubric_train.jsonl \
    --eval  data/rubric_eval.jsonl \
    --out-dir out/rubric-v1 \
    --max-seq-len 2560 \
    --epochs 3 --grad-accum 16

## 3. Çıktı

Adapter `out/rubric-v1/` altında, birkaç on MB. Bu koşu **Save Version** ile
kaydedilmeli: `rubric-eval` notebook'u adapter'ı `kernel_sources` üzerinden
buradan alıyor.

Loss sonuç değil. Sonuç `rubric-eval`'in ürettiği sayılar.

In [ ]:
f = "out/rubric-v1/train_metrics.json"
if os.path.exists(f):
    print(json.dumps(json.load(open(f)), indent=2, ensure_ascii=False))

!du -sh out/rubric-v1 2>/dev/null
!ls -la out/rubric-v1